# Camera Discovery — Harvest HLS-only Test

This notebook tests extraction-only harvesting of HLS camera/media URLs. It focuses on `.m3u8` discovery, source-row diagnostics, blind-search behavior, structured endpoint extraction, and harvest output summaries.

It does **not** run target resolution, geocoding, validation, trust gates, GeoJSON/map generation, or review ZIP creation. Use this notebook when the primary question is: *can harvest mode find HLS URLs?*

Expected runtime is moderate. The routine command avoids intermediate debug files. The optional debug/stress cell can create very large JSONL files and should be used only when troubleshooting extraction or dedupe.


## Setup

This notebook installs the repository code and runs the public CLI. It does not patch source files from the notebook.

Default behavior clones the `dev` branch. In Colab, restart the runtime after dependency installation if package imports behave unexpectedly.


In [ ]:
# Repository setup for Google Colab / notebook execution.
# Change REPO_BRANCH or REPO_URL if testing a fork/PR branch.
REPO_BRANCH = "dev"
REPO_URL = "https://github.com/dshipley71/camera-discovery.git"
REPO_DIR = "/content/camera-discovery"

from pathlib import Path
repo_dir = Path(REPO_DIR)
if not repo_dir.exists():
    !git clone -b "{REPO_BRANCH}" "{REPO_URL}" "{REPO_DIR}"
else:
    print(f"Repository already exists at {repo_dir}. Keeping existing checkout.")
%cd {REPO_DIR}
%pip install -e .[cloakbrowser] --no-build-isolation


## Credentials

In [ ]:
# Ollama Cloud / LLM credential setup.
# This avoids printing secrets. Configure OLLAMA_API_KEY in Colab: left sidebar > Secrets.
import os

try:
    from google.colab import userdata  # type: ignore
    OLLAMA_API_KEY = userdata.get('OLLAMA_API_KEY')
except Exception:
    OLLAMA_API_KEY = os.environ.get('OLLAMA_API_KEY')

if OLLAMA_API_KEY:
    os.environ['OLLAMA_API_KEY'] = OLLAMA_API_KEY
    os.environ.setdefault('CAMERA_DISCOVERY_LLM_PROVIDER', 'ollama-cloud')
    os.environ.setdefault('CAMERA_DISCOVERY_LLM_MODEL', 'gemma4:31b-cloud')
    print('Loaded OLLAMA_API_KEY from Colab userdata/environment')
else:
    print('OLLAMA_API_KEY not found. LLM-backed stages may fail unless another provider is configured.')

# Keep these visible so output records the provider/model, but never print the key.
print('LLM provider:', os.environ.get('CAMERA_DISCOVERY_LLM_PROVIDER', '(default from config)'))
print('LLM model:', os.environ.get('CAMERA_DISCOVERY_LLM_MODEL', '(default from config)'))


## Smoke tests

In [ ]:
# CLI and public import smoke tests.
import subprocess, sys

def run_cmd(cmd, *, env=None, check=True):
    print("\n$", " ".join(str(part) for part in cmd))
    result = subprocess.run([str(part) for part in cmd], env=env, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {' '.join(map(str, cmd))}")
    return result

run_cmd(['camera-discovery', '--help'])
run_cmd(['camera-discovery', 'run', '--help'])
run_cmd(['camera-discovery', 'harvest-urls', '--help'])

from camera_discovery.services.discovery_engine import CandidateDiscoveryEngine
from camera_discovery.services.harvest_engine import CameraUrlHarvestEngine
import camera_discovery.cli
print('camera-discovery imports OK')


## Notebook-only helper functions

In [ ]:
# Notebook-only inspection helpers. These intentionally live in the notebook, not src/.
from __future__ import annotations

import json
import os
import shutil
import subprocess
from collections import Counter
from pathlib import Path
from urllib.parse import urlparse


def read_json(path):
    path = Path(path)
    if not path.exists():
        print(f"Missing: {path}")
        return None
    return json.loads(path.read_text(encoding='utf-8'))


def iter_jsonl(path, limit=None):
    path = Path(path)
    if not path.exists():
        return
    with path.open('r', encoding='utf-8') as f:
        for idx, line in enumerate(f):
            if limit is not None and idx >= limit:
                break
            line = line.strip()
            if not line:
                continue
            yield json.loads(line)


def count_jsonl(path):
    path = Path(path)
    if not path.exists():
        return 0
    with path.open('r', encoding='utf-8') as f:
        return sum(1 for line in f if line.strip())


def top_hosts(path, url_field='url', limit=15):
    counts = Counter()
    for row in iter_jsonl(path):
        url = row.get(url_field) or row.get('stream_url') or row.get('source_url') or ''
        host = urlparse(url).netloc.casefold() or '(missing-host)'
        counts[host] += 1
    return counts.most_common(limit)


def media_counts(path):
    counts = Counter()
    for row in iter_jsonl(path):
        counts[row.get('media_type') or row.get('camera_type') or '(missing)'] += 1
    return dict(counts)


def scope_counts(path):
    counts = Counter()
    for row in iter_jsonl(path):
        counts[row.get('scope_status') or row.get('properties', {}).get('scope_status') or '(missing)'] += 1
    return dict(counts)


def print_json(path, keys=None):
    data = read_json(path)
    if data is None:
        return None
    if keys:
        data = {key: data.get(key) for key in keys}
    print(json.dumps(data, indent=2, sort_keys=True)[:12000])
    return data


def list_existing(paths):
    for path in paths:
        path = Path(path)
        print(f"{path}: {'exists' if path.exists() else 'missing'}" + (f" ({path.stat().st_size:,} bytes)" if path.exists() and path.is_file() else ''))


def package_output(output_dir, zip_name=None):
    output_dir = Path(output_dir)
    if zip_name is None:
        zip_name = str(output_dir).rstrip('/').replace('/', '_') + '.zip'
    zip_base = Path(zip_name).with_suffix('')
    archive = shutil.make_archive(str(zip_base), 'zip', root_dir=str(output_dir))
    print('Created archive:', archive)
    try:
        from google.colab import files  # type: ignore
        files.download(archive)
    except Exception:
        print('Download helper unavailable outside Colab. Archive remains at:', archive)
    return archive


def run_cli(cmd, *, env_overrides=None, check=True):
    env = os.environ.copy()
    if env_overrides:
        env.update({k: str(v) for k, v in env_overrides.items()})
    return run_cmd(cmd, env=env, check=check)


## Browser backend behavior

These notebooks default to browser capture disabled for structured endpoint / HLS workflows because the useful camera records usually come from static pages and JSON endpoints. Enable browser capture only when testing dynamic pages.

The cells below make the selected backend visible. They do not fake browser success.


In [ ]:
# Browser backend configuration for this notebook.
# For routine HLS/structured-endpoint tests, keep browser capture disabled.
BROWSER_BACKEND = "playwright"  # change to "cloakbrowser" when intentionally testing that backend
DISABLE_BROWSER_CAPTURE_ENV = {"CAMERA_DISCOVERY_ENABLE_BROWSER_CAPTURE": "false", "CAMERA_DISCOVERY_BROWSER_BACKEND": BROWSER_BACKEND}
print('Browser backend selected:', BROWSER_BACKEND)
print('Browser capture default for this notebook:', DISABLE_BROWSER_CAPTURE_ENV['CAMERA_DISCOVERY_ENABLE_BROWSER_CAPTURE'])
print('To test dynamic browser capture, remove --disable-browser-capture for harvest and set CAMERA_DISCOVERY_ENABLE_BROWSER_CAPTURE=true for run.')


## Run HLS-only harvest

In [ ]:
# Routine HLS-only harvest. Completion-aware guard checks harvest_summary.json and harvest_handoff.json.
QUERY = "California traffic cameras"
HARVEST_DIR = Path('runs/harvest-california-hls')
RERUN_HARVEST = False

expected = [HARVEST_DIR / 'harvest_summary.json', HARVEST_DIR / 'harvest_handoff.json', HARVEST_DIR / 'camera_urls.jsonl']
if RERUN_HARVEST and HARVEST_DIR.exists():
    shutil.rmtree(HARVEST_DIR)

if not all(path.exists() for path in expected):
    print('Running visible harvest CLI command. Output will stream below.')
    !camera-discovery harvest-urls "{QUERY}"       --output-dir "{HARVEST_DIR}"       --discovery-mode both       --max-search-queries 12       --max-search-results-per-query 25       --max-source-rows 1000       --max-pages-per-source 10       --max-urls 0       --media .m3u8       --disable-browser-capture       --progress-style plain
else:
    print('Skipping harvest: completion artifacts already exist. Set RERUN_HARVEST=True to rerun.')


## Inspect HLS harvest

In [ ]:
# Inspect harvest outputs.
HARVEST_DIR = Path(HARVEST_DIR)
print('Harvest directory:', HARVEST_DIR)
list_existing([
    HARVEST_DIR / 'harvest_summary.json',
    HARVEST_DIR / 'harvest_handoff.json',
    HARVEST_DIR / 'camera_urls.txt',
    HARVEST_DIR / 'camera_urls.csv',
    HARVEST_DIR / 'camera_urls.jsonl',
    HARVEST_DIR / 'camera_records.jsonl',
    HARVEST_DIR / 'camera_media_assets.jsonl',
    HARVEST_DIR / 'discovered_endpoints.jsonl',
    HARVEST_DIR / 'logs' / 'source_rows_summary.json',
    HARVEST_DIR / 'logs' / 'harvest_blind_search_diagnostics.jsonl',
    HARVEST_DIR / 'logs' / 'harvest_errors.jsonl',
])

summary = print_json(HARVEST_DIR / 'harvest_summary.json', keys=[
    'raw_count', 'unique_count', 'written_count', 'by_media_type', 'by_source_provider', 'by_source_host',
    'camera_record_count', 'media_asset_count', 'discovered_endpoint_count', 'warnings'
])
print('\nSource row summary:')
print_json(HARVEST_DIR / 'logs' / 'source_rows_summary.json')
print('\nHandoff manifest:')
print_json(HARVEST_DIR / 'harvest_handoff.json')
print('\nMedia counts from camera_urls.jsonl:', media_counts(HARVEST_DIR / 'camera_urls.jsonl'))
print('\nTop URL hosts from camera_urls.jsonl:', top_hosts(HARVEST_DIR / 'camera_urls.jsonl'))
print('\nJSONL counts:')
for name in ['camera_urls.jsonl', 'camera_records.jsonl', 'camera_media_assets.jsonl', 'discovered_endpoints.jsonl', 'harvest_camera_inventory.jsonl']:
    print(name, count_jsonl(HARVEST_DIR / name))
print('\nSample camera_urls.jsonl rows:')
for row in iter_jsonl(HARVEST_DIR / 'camera_urls.jsonl', limit=5):
    print(json.dumps(row, indent=2)[:2000])


## Optional expensive debug/stress run

This writes raw/unique/media-filtered intermediate JSONL files. They can be hundreds of MB or larger. Do not run for routine checks.

In [ ]:
# Optional expensive debug run. Disabled by default.
RUN_EXPENSIVE_INTERMEDIATE_DEBUG = False
DEBUG_DIR = Path('runs/harvest-california-hls-debug-intermediate')
if RUN_EXPENSIVE_INTERMEDIATE_DEBUG:
    if DEBUG_DIR.exists():
        shutil.rmtree(DEBUG_DIR)
    print('Running visible expensive harvest debug CLI command. Output will stream below.')
    !camera-discovery harvest-urls "{QUERY}"       --output-dir "{DEBUG_DIR}"       --discovery-mode both       --max-search-queries 20       --max-search-results-per-query 40       --max-source-rows 2000       --max-pages-per-source 15       --max-urls 0       --media .m3u8       --disable-browser-capture       --write-intermediate-records       --progress-style plain
else:
    print('Skipping expensive intermediate debug run.')


## Package outputs

In [ ]:
# Optional: package and download outputs.
# Run this after the workflow completes.
package_output('runs/harvest-california-hls')
